# Lesson 30: LangChain — building blocks for LLM apps 🔗
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain what LangChain is for and its main building blocks
2. Use **chat models**, **prompt templates** and **output parsers**
3. Connect them into **chains** with the `|` pipe (LCEL), including parallel chains
4. Get **structured output** (Pydantic), add **conversation memory**, and preview **tools and agents**

**⏱️ Timing:** models & prompts 15' | chains 15' | structured output 10' | memory 10' | tools preview 10'

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 🧱 An analogy: Lego Technic for AI apps
Calling an LLM once is easy. Real apps need much more: templates, several steps, parsing answers into data, memory, tools, swapping between OpenAI, Claude, Gemini or local models…
**LangChain** (released 2022) is an open-source framework with standard pieces that snap together:
| Piece | Job |
|---|---|
| **Chat model** | Talks to any LLM with the same code (`ChatOllama`, `ChatOpenAI`, `ChatAnthropic`, …) |
| **Prompt template** | A fill-in-the-blanks prompt |
| **Output parser** | Turns text into a string, JSON or a Python object |
| **Chain** (LCEL) | Pieces joined with `\|`, like a factory conveyor belt |
| **Memory / history** | Keeps the conversation |
| **Tools & agents** | Lets the model call functions (Lessons 34–36) |

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install langchain langchain-ollama langchain-core
import warnings; warnings.filterwarnings("ignore")
from langchain_ollama import ChatOllama
llm = ChatOllama(model=MODEL, temperature=0.3)

reply = llm.invoke("Give me one fun fact about Saturn in one sentence.")
print(type(reply).__name__, "→", reply.content)

> 💡 To switch to another provider, you'd change just this one line, e.g. `ChatAnthropic(model=...)` or `ChatOpenAI(model=...)` with a key from Colab Secrets (Lesson 25). The rest of the notebook stays the same.

## Part 1: Messages and prompt templates 📝

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
print(llm.invoke([SystemMessage("You only answer in rhyming couplets."),
                  HumanMessage("What is a volcano?")]).content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
explain = ChatPromptTemplate.from_messages([
    ("system", "You are a {subject} teacher. Explain things for a {age}-year-old in at most {sentences} sentences."),
    ("user", "{question}"),
])
filled = explain.invoke({"subject": "biology", "age": 12, "sentences": 2, "question": "What does a heart do?"})
print(filled.to_messages())

## Part 2: Chains with `|` (LCEL) ⛓️
**LCEL** = LangChain Expression Language. `prompt | llm | parser` means "fill the template, send to the model, clean the output".

In [ ]:
from langchain_core.output_parsers import StrOutputParser
chain = explain | llm | StrOutputParser()

print(chain.invoke({"subject": "physics", "age": 12, "sentences": 2, "question": "Why do things fall?"}))

In [ ]:
# Every chain gets .batch() and .stream() for free
questions = [{"subject": "chemistry", "age": 14, "sentences": 1, "question": q}
             for q in ["What is an acid?", "Why does iron rust?", "What is a molecule?"]]
for q, a in zip(questions, chain.batch(questions)):
    print(f"❓ {q['question']}\n   {a}")

print("\nStreaming: ", end="")
for token in chain.stream({"subject": "geography", "age": 10, "sentences": 2, "question": "What is a glacier?"}):
    print(token, end="", flush=True)

In [ ]:
# Chains can feed other chains: write a quiz question, then answer-check it
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

make_q = ChatPromptTemplate.from_template("Write ONE short quiz question about {topic}. Only the question.") | llm | StrOutputParser()
answer_q = ChatPromptTemplate.from_template("Answer this quiz question in one sentence: {question}") | llm | StrOutputParser()

quiz_chain = {"question": make_q} | RunnablePassthrough.assign(answer=answer_q)
print(quiz_chain.invoke({"topic": "the water cycle"}))

In [ ]:
# Run several chains at the same time on the same input
helper = RunnableParallel(
    simple=ChatPromptTemplate.from_template("Explain {idea} to a 7-year-old in one sentence.") | llm | StrOutputParser(),
    expert=ChatPromptTemplate.from_template("Explain {idea} to a university student in one sentence.") | llm | StrOutputParser(),
    emoji=ChatPromptTemplate.from_template("Summarise {idea} using only 5 emojis.") | llm | StrOutputParser(),
)
for k, v in helper.invoke({"idea": "photosynthesis"}).items():
    print(f"{k:7}: {v}")

### 🧩 Exercise 1
Build a **translation chain**: a template with `{text}` and `{language}`, then `| llm | StrOutputParser()`. Use `.batch()` to translate one sentence into 3 languages.

## Part 3: Structured output — answers as data 📦
Apps need data, not paragraphs. Describe the shape with **Pydantic**, and LangChain makes the model fill it in.

In [ ]:
from pydantic import BaseModel, Field

class Flashcard(BaseModel):
    term: str = Field(description="the key word")
    definition: str = Field(description="a definition under 20 words")
    example: str = Field(description="a real-life example")
    difficulty: int = Field(description="1 (easy) to 5 (hard)")

card_maker = llm.with_structured_output(Flashcard)
card = card_maker.invoke("Make a flashcard about the word 'evaporation'.")
print(card)
print("\nJust the definition:", card.definition)

In [ ]:
class Quiz(BaseModel):
    topic: str
    cards: list[Flashcard]

quiz = llm.with_structured_output(Quiz).invoke("Make 3 flashcards about the solar system.")
for c in quiz.cards:
    print(f"🃏 {c.term:12} (level {c.difficulty}) — {c.definition}")

## Part 4: Memory — remembering the conversation 💭
LLMs don't remember (Lesson 25). LangChain stores the **message history** and sends it each time.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

tutor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly maths tutor. Keep replies short."),
    MessagesPlaceholder("history"),
    ("user", "{input}"),
])
histories = {}
def get_history(session_id):
    return histories.setdefault(session_id, InMemoryChatMessageHistory())

tutor = RunnableWithMessageHistory(tutor_prompt | llm | StrOutputParser(), get_history,
                                   input_messages_key="input", history_messages_key="history")
cfg = {"configurable": {"session_id": "student-1"}}
for msg in ["Hi! My name is Aroha and I'm learning fractions.", "What is 1/2 + 1/4?", "What's my name, and what am I learning?"]:
    print(f"🧑 {msg}\n🤖 {tutor.invoke({'input': msg}, config=cfg)}\n")
print("Messages stored:", len(histories["student-1"].messages))

### 🧩 Exercise 2
1. Start a **new** session id (`"student-2"`) and ask "What's my name?". What happens, and why?
2. Long histories cost more tokens. Suggest two ways to keep memory short. (Keep only the last N messages; summarise old messages.)

## Part 5: Preview — tools and agents 🛠️
LangChain can turn any Python function into a **tool** and build an **agent** that decides when to call it (the ReAct idea from Lesson 29, done for you).

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def term_dates(term: int) -> str:
    """Get the start and end dates of a school term (1 to 4)."""
    dates = {1: "3 Feb – 11 Apr", 2: "28 Apr – 4 Jul", 3: "21 Jul – 26 Sep", 4: "13 Oct – 16 Dec"}
    return dates.get(term, "No such term")

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

agent = create_agent(llm, tools=[term_dates, multiply], system_prompt="You are a school office assistant. Use tools when needed.")
result = agent.invoke({"messages": [{"role": "user", "content": "When does term 3 start? Also, what is 64 times 185?"}]})
for m in result["messages"]:
    print(f"{type(m).__name__:12}", getattr(m, "tool_calls", None) or m.content)

### 🚀 Challenge
Build a "**homework planner**" chain: input = a list of subjects and due dates → structured output = a `Plan` with a list of `Task(subject, day, minutes)` → print it as a timetable.

## 📌 Summary
- LangChain = standard, swappable building blocks for LLM apps
- **prompt | llm | parser** chains (LCEL), with `.invoke`, `.batch`, `.stream`, and parallel branches
- **`with_structured_output`** turns answers into Python objects
- **Message history** gives conversations memory; **tools + `create_agent`** let the model act